In [3]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [ ]:
def preparer_donnees_climat():
   
    base_path = r"Data"
    
   
    df_grille = pd.read_csv("Data\csv_final\grid_portugal_1km.csv")
    print(f"Grille: {len(df_grille)} points")
    
    df_complet = df_grille[['longitude', 'latitude']].copy()
    
    
    dossiers_climat = {
        'Precipitation': 'prec',
        'Tmin': 'tmin', 
        'Tmax': 'tmax'
    }
    
    for dossier, prefixe in dossiers_climat.items():
        chemin_dossier = os.path.join(base_path, "climate", dossier)
        fichiers_tif = glob.glob(os.path.join(chemin_dossier, "*.tif"))
        
        print(f"\n {dossier}: {len(fichiers_tif)} fichiers")
        
        for tif_path in sorted(fichiers_tif):
            nom_fichier = os.path.basename(tif_path)
            mois = nom_fichier.split('_')[-1].replace('.tif', '')
            nom_colonne = f"{prefixe}_{mois}"
            
            print(f"{mois}...", end=" ")
            
            
            valeurs = extraction_vectorisee(tif_path, df_grille)
            df_complet[nom_colonne] = valeurs
            
            valides = np.sum(~np.isnan(valeurs))
            print(f"({valides} valides)")

    output_csv = os.path.join(base_path, "climate\donnees_climat_1km.csv")
    df_complet.to_csv(output_csv, index=False)
    
    return df_complet


def extraction_vectorisee(tif_path, df_points):
    with rasterio.open(tif_path) as src:
        
        rows, cols = rasterio.transform.rowcol(
            src.transform, 
            df_points['longitude'].values, 
            df_points['latitude'].values
        )
        
      
        mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)
        
        
        valeurs = np.full(len(df_points), np.nan)
        
        if np.any(mask):
            
            data = src.read(1)
            valeurs_valides = data[rows[mask], cols[mask]]
            
            
            valeurs[mask] = valeurs_valides
            
            
            valeurs[valeurs == src.nodata] = np.nan
    
    return valeurs


donnees_rapides = preparer_donnees_climat()

Grille: 89198 points

 Precipitation: 12 fichiers
2024-01... (89192 valides)
2024-02... (89192 valides)
2024-03... (89192 valides)
2024-04... (89192 valides)
2024-05... (89192 valides)
2024-06... (89192 valides)
2024-07... (89192 valides)
2024-08... (89192 valides)
2024-09... (89192 valides)
2024-10... (89192 valides)
2024-11... (89192 valides)
2024-12... (89192 valides)

 Tmin: 12 fichiers
2024-01... (89192 valides)
2024-02... (89192 valides)
2024-03... (89192 valides)
2024-04... (89192 valides)
2024-05... (89192 valides)
2024-06... (89192 valides)
2024-07... (89192 valides)
2024-08... (89192 valides)
2024-09... (89192 valides)
2024-10... (89192 valides)
2024-11... (89192 valides)
2024-12... (89192 valides)

 Tmax: 12 fichiers
2024-01... (89192 valides)
2024-02... (89192 valides)
2024-03... (89192 valides)
2024-04... (89192 valides)
2024-05... (89192 valides)
2024-06... (89192 valides)
2024-07... (89192 valides)
2024-08... (89192 valides)
2024-09... (89192 valides)
2024-10... (89192 v